# The OptiPFair Series #2: Healing the Golden Scar

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/principia-agentica/principia-agentica/blob/main/articles/032226-optipfair/optipfair_series_2.ipynb)


In our [second article](https://principia-agentica.com/blog/2026/03/22/healing-the-golden-scar-optipfair/) of the OptiPFair Series, we explored two massive architectural evolutions in Pere Martra's model optimization library:

1. **Hardware-Aware Alignment (`expansion_divisor`)**: Ensuring pruned tensors remain perfectly divisible by hardware-friendly numbers (like 64) for Tensor Core efficiency.
2. **Data-Driven Pruning (`dataloader`)**: Shifting from static weight analysis to dynamic, context-aware pruning using the Peak-to-Peak Magnitude (PPM) method.

This notebook is the practical proof of concept. We will take `meta-llama/Llama-3.2-1B`, expose its "golden scar" through naive static pruning, and then heal it by building a highly specialized coding engine using `CodeAlpaca`.

## 1. Setup and Baseline
First, we install the necessary tools and load the base model.

In [ ]:
!pip install -q optipfair transformers datasets torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import optipfair as opf

MODEL_ID = "meta-llama/Llama-3.2-1B"

print(f"Loading baseline model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Ensure pad token is set for dataloader batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the model on CPU for the PoC (or CUDA if available)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16).to(device)

# Let's inspect the original intermediate size of the MLP layer
original_intermediate_size = model.model.layers[0].mlp.gate_proj.out_features
print(f"\nOriginal Llama-3.2-1B Intermediate Size: {original_intermediate_size}")
print(f"Divisible by 64? {original_intermediate_size % 64 == 0}")

## 2. The Baseline (The Jagged Edge)
Let's simulate the "golden scar." We will perform a naive, static width prune (cutting 40% of the neurons) without the `expansion_divisor`. 

Notice how the resulting tensor size fractures the hardware alignment.

> 💡 **Architectural Note: The Llama GLU Compression Quirks**
> 
> You might wonder why we are pruning by a massive `40%`. 
> 
> Llama models define their Multi-Layer Perceptron (MLP) width (`intermediate_size`) using a highly optimized SwiGLU structure. This native dimension is often compressed far below the standard theoretical calculation `(hidden_size * 4 * 2 / 3)`.
> 
> If we ask OptiPFair to prune a Llama model by only `20%`, the algorithm calculates its theoretical mathematical baseline, applies the 20% cut, and the resulting number can sometimes end up being *larger* than Llama's already heavily-compressed native `intermediate_size`! Thus, your "pruned" model actually expands.
> 
> To guarantee a true parameter reduction against Llama's already aggressive baseline, we must cut deeper. Hence, `pruning_percentage=40`.

In [ ]:
print("Building baseline... Static pruning (40%) without hardware alignment.")

# Clone model to avoid modifying the base instance for later steps
import copy
static_model = copy.deepcopy(model)

static_pruned, static_stats = opf.prune_model(
    model=static_model,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW", # PPM method (static fallback without dataloader)
    pruning_percentage=40,
    show_progress=False,
    return_stats=True
)

static_intermediate_size = static_pruned.model.layers[0].mlp.gate_proj.out_features
print(f"\nJagged Intermediate Size: {static_intermediate_size}")
print(f"Divisible by 64? {static_intermediate_size % 64 == 0}")
print("Result: Tensor Cores will waste cycles padding this dimension.")



## 3. Calibration Data
To build a true specialist, we must provide a dataset representing its target domain. We will load a subset of the `sahil2801/CodeAlpaca-20k` dataset. 

By passing this through the dataloader, the PPM method will measure which neurons actually light up when processing Python code. Neurons irrelevant to this domain will be pruned.

In [ ]:
from datasets import load_dataset
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

print("Loading the CodeAlpaca calibration dataset...")
dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train[:100]") # Tiny slice for PoC

def tokenize_function(examples):
    # Combine instruction and output for calibration context
    texts = [f"{i} {o}" for i, o in zip(examples["instruction"], examples["output"])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")

print("Tokenizing calibration data...")
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
tokenized_dataset.set_format("torch")

# Create the DataLoader
# OptiPFair expects a TensorDataset yielding (input_ids, attention_mask)
import torch
input_ids = tokenized_dataset['input_ids']
if not isinstance(input_ids, torch.Tensor):
    input_ids = torch.stack([torch.tensor(x) for x in input_ids])
attention_mask = tokenized_dataset['attention_mask']
if not isinstance(attention_mask, torch.Tensor):
    attention_mask = torch.stack([torch.tensor(x) for x in attention_mask])
tensor_dataset = TensorDataset(input_ids, attention_mask)
dataloader = DataLoader(tensor_dataset, batch_size=4)


## 4. Data-Driven Pruning with Hardware Alignment
Now, we apply the hybrid, data-driven prune. We run the hybrid, data-driven prune using our `dataloader` AND we apply the `expansion_divisor=64` to mathematically snap the resulting layers back into hardware alignment.

In [ ]:
import copy
specialized_base = copy.deepcopy(model)

print("Calibrating hybrid resonance and snapping to grid...")

specialized_model, specialized_stats = opf.prune_model(
    model=specialized_base,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW", # Invokes Hybrid PPM with dataloader
    pruning_percentage=40,
    expansion_divisor=64,          # HARDWARE ALIGNMENT
    dataloader=dataloader,         # DATA-DRIVEN CALIBRATION
    show_progress=True,
    return_stats=True
)

specialized_intermediate_size = specialized_model.model.layers[0].mlp.gate_proj.out_features
print(f"\nHealed Intermediate Size: {specialized_intermediate_size}")
print(f"Divisible by 64? {specialized_intermediate_size % 64 == 0}")
print("Result: The pruned dimensions are perfectly aligned to hardware constraints.")

## 5. Benchmarking & Inference
To truly understand the value of this approach, we must measure the results. We will compare:
1. **The Original Model**
2. **The Statically Pruned Model** (The baseline)
3. **The Data-Driven Specialized Model** (The healed scar)

We will measure parameter size reduction, tokens-per-second (TPS) performance, and evaluate the quality of a coding prompt.

In [ ]:
import time
import torch

def benchmark_model(name, eval_model, tokenizer, prompt, device):
    eval_model.eval()
    print(f"\n{'='*40}")
    print(f" 🏎️ Benchmarking: {name}")
    print(f"{'='*40}")
    
    # 1. Measure Parameters (Size Reduction)
    params = sum(p.numel() for p in eval_model.parameters())
    print(f"Parameters: {params:,}")
    
    # Prepare inputs
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Warmup pass (important for accurate GPU timing)
    with torch.no_grad():
        _ = eval_model.generate(**inputs, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)
        
    # 2. Timed generation (Performance Improvement)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.time()
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs, 
            max_new_tokens=60, 
            temperature=0.2, # Lower temperature for more deterministic coding output
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.time()
    
    generated_tokens = outputs[0].shape[0] - inputs.input_ids.shape[1]
    generation_time = end_time - start_time
    tps = generated_tokens / generation_time
    
    print(f"Speed: {tps:.2f} tokens/sec ({generation_time:.2f}s total)")
    print(f"\n--- Output ---")
    
    # 3. Output Quality
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    
    return params, tps

# A standard Python coding prompt
test_prompt = "def calculate_factorial(n):\n    \"\"\"Calculates the factorial of a number.\"\"\""

# Run Benchmarks
bench_original = benchmark_model("Original Llama-3.2-1B", model, tokenizer, test_prompt, device)
bench_static = benchmark_model("Static Pruned (40% Naive)", static_pruned, tokenizer, test_prompt, device)
bench_specialist = benchmark_model("Data-Driven Specialist (40% + HW Aligned)", specialized_model, tokenizer, test_prompt, device)

# Final Analytical Summary
print(f"\n{'*'*60}")
print(f" 🏆 FINAL BENCHMARK COMPARISON")
print(f"{'*'*60}")

def compare(name, base_stats, new_stats):
    size_reduction = 100 - (new_stats[0] / base_stats[0] * 100)
    speedup = new_stats[1] / base_stats[1]
    print(f"\n{name}:")
    print(f"  - Size: {size_reduction:.2f}% smaller ({base_stats[0]:,} -> {new_stats[0]:,})")
    print(f"  - Speed: {speedup:.2f}x faster ({base_stats[1]:.2f} TPS -> {new_stats[1]:.2f} TPS)")

# 1. Original vs Static
compare("A vs B (Original -> Static Pruned)", bench_original, bench_static)

# 2. Original vs Specialist
compare("A vs C (Original -> Specialist)", bench_original, bench_specialist)

# 3. Static vs Specialist
compare("B vs C (Static Pruned -> Specialist)", bench_static, bench_specialist)


## 6. Conclusion: The Golden Scar Revealed

If you review the generated text and the final benchmark metrics, the value of OptiPFair's evolution becomes mathematically and practically undeniable:

1. **Output Quality (The Brain Damage of Naive Pruning):** 
   - The **Original** 1.23B model wrote a coherent recursive factorial.
   - The **Static Pruned** model (913M params) suffered catastrophic logic loss, returning `n` and printing `"Hello"` endlessly.
   - The **Data-Driven Specialist** (914M params), despite being pruned by the exact same percentage, retained its coding logic and successfully wrote a recursive factorial. It preserved the capability we calibrated it for while shedding generalist weights.

2. **The Hardware Alignment Proof (The Math):**
   - We successfully compressed the model by **~26%** (saving over a quarter of a billion parameters) while achieving a **~1.4x speedup** in inference.
   - Notice that the **Specialist** model is technically slightly *larger* than the Static one (due to the `expansion_divisor=64` padding it up to the nearest multiple). Yet, because it is perfectly aligned to the hardware grid, it doesn't waste compute cycles on unaligned tensor math, maintaining parity or beating the jagged static model in speed.